In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris

import os
current_directory = os.getcwd()


In [ ]:
print(current_directory)
covid_path = "../data/hrv-covid19/data/"

# scalesdf = pd.read_csv("scales_description.csv")
hrvdf = pd.read_csv(covid_path + "hrv_measurements.csv")
hrvdf.head()

/content


,user_code,rr_code,measurement_datetime,time_of_day,bpm,meanrr,mxdmn,sdnn,rmssd,pnn50,...,lf,hf,vlf,lfhf,total_power,how_feel,how_mood,how_sleep,tags,rr_data
0,007b8190cf,10489a6aea,2020-04-21 21:23:08,morning,75,795.90,0.12,45.802,54.174,15.15,...,508.0,1076.0,267.0,0.472,1851.0,0,-1,NaN,COVID-19; Workout; Sex; Hobby; Studying; Sleep...,"819,1008,831,847,785,778,866,839,801,793,846,8..."
1,007b8190cf,9610d4d4dc,2020-04-26 11:19:25,morning,70,858.00,0.11,32.889,33.022,16.16,...,409.0,310.0,176.0,1.319,895.0,0,0,0.0,NaN,"888,775,811,883,890,894,894,899,893,889,890,83..."
2,013f6d3e5b,f3de056155,2020-05-15 04:14:21,night,83,724.10,0.17,54.811,65.987,17.17,...,432.0,881.0,194.0,0.490,1507.0,-1,-2,NaN,COVID-19; Fast/Diet; Hungry; Tired; Fever; I c...,"694,832,642,801,751,716,737,742,773,760,701,73..."
3,013f6d3e5b,b04489e32f,2020-05-19 03:06:02,night,75,802.64,0.20,72.223,70.039,22.22,...,814.0,1487.0,1719.0,0.547,4020.0,0,0,NaN,NaN,"821,817,771,805,833,788,747,724,792,825,775,75..."
4,01bad5a519,ac52c706c6,2019-12-31 09:07:43,morning,78,768.07,0.10,29.650,21.196,4.04,...,489.0,128.0,96.0,3.820,713.0,0,0,0.0,NaN,"741,740,734,737,740,731,751,747,745,728,747,76..."


# Objective of Data Exploration


I want to explore HRF data. We want to first investigate the features and statistical data of each of these datasets. I want to find potential correlations between each of these datasets

In [6]:
print("Heart Rate Variabilitiy", hrvdf.shape, hrvdf.columns, "INFO \n",hrvdf.info(), "Description",hrvdf.describe(), hrvdf.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3245 entries, 0 to 3244
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   user_code             3245 non-null   object 
 1   rr_code               3245 non-null   object 
 2   measurement_datetime  3245 non-null   object 
 3   time_of_day           3245 non-null   object 
 4   bpm                   3245 non-null   int64  
 5   meanrr                3245 non-null   float64
 6   mxdmn                 3245 non-null   float64
 7   sdnn                  3245 non-null   float64
 8   rmssd                 3245 non-null   float64
 9   pnn50                 3245 non-null   float64
 10  mode                  3245 non-null   float64
 11  amo                   3245 non-null   float64
 12  lf                    3245 non-null   float64
 13  hf                    3245 non-null   float64
 14  vlf                   3245 non-null   float64
 15  lfhf                 

The first step is to learn more about these features. R-R intervals refers to the time duration between R-waves or spikes on an ECG for ventricular contraction.
We have a lot of diverse metrics collected about RR frequency: mean, max diff min, standard deviation, nn50 (percentage of normal-tonormal intervals that differ more than 50), mode, modal Amplitude, etc. We also get the power of low frequency and high frequency signals in milliseconds squared? I recognize my lack of domain knowledge in HRV data and will choose not to utilize the HR data above.


# Wearables Profiles

I want to build a profile of each unique user based on their behaviors. Lets start by documenting if the user is a repeating user in the original wearables database. This will allow us to easily filter the profiles we want to build in the future.

In [ ]:
wdf = pd.read_csv(covid_path + "wearables.csv")
print("Wearables", wdf.shape, wdf.columns, "INFO \n",wdf.info(), "Description",wdf.describe(), wdf.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3098 entries, 0 to 3097
Data columns (total 18 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   user_code                        3098 non-null   object 
 1   day                              3098 non-null   object 
 2   resting_pulse                    1515 non-null   float64
 3   pulse_average                    2089 non-null   float64
 4   pulse_min                        2089 non-null   float64
 5   pulse_max                        2089 non-null   float64
 6   average_spo2_value               40 non-null     float64
 7   body_temperature_avg             65 non-null     float64
 8   stand_hours_total                531 non-null    float64
 9   steps_count                      1968 non-null   float64
 10  distance                         1869 non-null   float64
 11  steps_speed                      1418 non-null   float64
 12  total_number_of_flig

In [15]:
len(wdf.user_code.unique())
# We get 79 unique users

79

In [48]:
# I want to collect the number of entries per unique user
user_counts =wdf.user_code.value_counts() # Some have 1 entry, some have multiple.

# Lets only use repeated users: > 10 entries
counts = user_counts[user_counts > 10]
len(counts) # Wonderful! There are 49, Let's get
print("User Count Mean", (counts.mean()), "User Count Standard Deviation", user_counts.std())


User Count Mean 61.59183673469388 User Count Standard Deviation 45.89499466761501


Let's look back at our Na data so that we can maximize our dataset. We have a lot of Na values, but we seem to have more complete pulse data (pulse avg, min, and max) as well as steps data (steps_count, distance, and steps_speed). Lets copy our dataset and try to build a new profiles dataframe to learn more about our users.

In [45]:
wdf["isRepeated"] = wdf.user_code.isin(user_counts.index)
wdf["isRepeated"]
print(wdf.head())

    user_code         day  resting_pulse  pulse_average  pulse_min  pulse_max  \
0  007b8190cf  2020-04-26            NaN           70.0       70.0       70.0   
1  01bad5a519  2020-02-12            NaN            NaN        NaN        NaN   
2  01bad5a519  2020-02-13            NaN            NaN        NaN        NaN   
3  01bad5a519  2020-02-15            NaN            NaN        NaN        NaN   
4  01bad5a519  2020-02-16            NaN            NaN        NaN        NaN   

   average_spo2_value  body_temperature_avg  stand_hours_total  steps_count  \
0                 NaN                   NaN                NaN          NaN   
1                 NaN                   NaN                NaN       8574.0   
2                 NaN                   NaN                NaN       7462.0   
3                 NaN                   NaN                NaN       2507.0   
4                 NaN                   NaN                NaN      10131.0   

   distance  steps_speed  total_number

Now lets

In [47]:
df = wdf.copy()
df = df[df["isRepeated"]] # Only use repeating users




    user_code         day  resting_pulse  pulse_average  pulse_min  pulse_max  \
1  01bad5a519  2020-02-12            NaN            NaN        NaN        NaN   
2  01bad5a519  2020-02-13            NaN            NaN        NaN        NaN   
3  01bad5a519  2020-02-15            NaN            NaN        NaN        NaN   
4  01bad5a519  2020-02-16            NaN            NaN        NaN        NaN   
5  01bad5a519  2020-02-17            NaN            NaN        NaN        NaN   

   average_spo2_value  body_temperature_avg  stand_hours_total  steps_count  \
1                 NaN                   NaN                NaN       8574.0   
2                 NaN                   NaN                NaN       7462.0   
3                 NaN                   NaN                NaN       2507.0   
4                 NaN                   NaN                NaN      10131.0   
5                 NaN                   NaN                NaN       8086.0   

   distance  steps_speed  total_number

In [79]:
profiles = pd.DataFrame()
profiles["user_code"] = user_counts[user_counts >10].keys()
profiles["user_count"] = user_counts[user_counts > 10].values
print(profiles.head(), profiles.tail())

    user_code  user_count
0  0d297d2410         170
1  6be5033971         162
2  fcf3ea75b0         153
3  c174f32d88         152
4  276ab22485         128      user_code  user_count
44  b325aa1406          13
45  cf7e50bcde          13
46  9b4a5ded49          12
47  29d2a4bb3a          11
48  2e94435085          11


In [80]:
# Now let's track average total calories burned per day per user
avg_daily = df.groupby("user_code")["total_calories_burned"].mean().reset_index()
avg_daily.rename(columns={"total_calories_burned": "avgDailyCalories"}, inplace=True)

profiles = profiles.merge(avg_daily, on="user_code")
print(profiles)

     user_code  user_count  avgDailyCalories
0   0d297d2410         170       2766.205882
1   6be5033971         162       1842.487654
2   fcf3ea75b0         153       2416.392157
3   c174f32d88         152       3884.631579
4   276ab22485         128       2585.859375
5   4985083f4d         127       1851.181102
6   a1c2e6b2eb         119       3366.521008
7   e8240b51a2         113       2474.292035
8   01bad5a519         113       2624.000000
9   9871ee5e7b         108       2295.261682
10  ebf2c3cb63         108       1949.000000
11  35c7355282         107       2757.168224
12  5d200bd1c6         101       1953.366337
13  295ed96279          83       1906.289157
14  b9b65b7a69          77       2258.000000
15  ad41d5b79c          73       2072.602740
16  71980b2daf          66       1683.000000
17  b523b4512b          64       2350.718750
18  d40dc56a36          60       2463.333333
19  aa036185e3          60       4841.566667
20  cdf7848d2b          59       2558.050847
21  fde848

Now let's build a function that adds average wearables dataset info to the profiles dataset

In [81]:
def avg_daily_something(profiles, something: str):
  avg_daily = df.groupby("user_code")[something].mean().reset_index()
  avg_daily.rename(columns={something: f"avg_{something}"}, inplace=True)
  return profiles.merge(avg_daily, on="user_code")

In [82]:
# create these columns (There is a better way to do this than returning
profiles= avg_daily_something(profiles, "steps_count")
profiles= avg_daily_something(profiles, "distance")
profiles=avg_daily_something(profiles, "steps_speed")

In [83]:
profiles

,user_code,user_count,avgDailyCalories,avg_steps_count,avg_distance,avg_steps_speed
0,0d297d2410,170,2766.205882,8587.292994,6815.333333,27.282821
1,6be5033971,162,1842.487654,7498.076336,5007.849624,23.107845
2,fcf3ea75b0,153,2416.392157,5316.529851,4038.857143,26.736881
3,c174f32d88,152,3884.631579,3749.774834,2527.472973,14.226061
4,276ab22485,128,2585.859375,894.277778,881.784314,24.600000
5,4985083f4d,127,1851.181102,7588.550725,5332.000000,7.392800
6,a1c2e6b2eb,119,3366.521008,9491.388889,8430.905660,17.093958
7,e8240b51a2,113,2474.292035,6369.043478,3807.153846,52.002500
8,01bad5a519,113,2624.000000,7109.415929,NaN,56.745432
9,9871ee5e7b,108,2295.261682,3105.481481,1844.096154,31.451852


## Summary

Amazing! We built a basic profiles dataset by grouping together daily wearables data by user id. This investigation allows us to learn more about the people's behavioral patterns using the wearables, rather than general daily data. Today, we learned about average daily calories, step count, distance walked, and step speed. We could use this data to find general correlations between step patterns and calories burnt. There are obvious deviations in the data already. For example: user 276ab22485 (row 4) burns about ~2585 calories despite averaging ~894 steps. Perhaps he swims or finds another way to stay active.